In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <호출(invoke) 출력>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 오픈AI의 대규모 언어 모델 설정
model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 주어진 주제에 대한 설명 요청
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧게 설명을 해주세요")
# 출력 파서 정의: AI 메시지의 출력 내용을 추출
parser = StrOutputParser()
# 프롬프트, 모델, 출력 파서를 체인으로 연결
chain = prompt | model | parser

# 응답을 호출
chain.invoke({"topic":"더블딥"})


'더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체 후 회복과정 중에 다시 경제가 둔화되는 현상을 의미합니다. 일반적으로 경제가 한 번 하강한 후 일시적으로 회복세를 보이다가 다시 심각한 불황으로 빠지는 경우를 지칭합니다. 이는 종종 지속적인 실업률 증가, 소비 감퇴, 또는 고용 시장의 회복 지연 등으로 인해 발생할 수 있습니다. 더블딥 경제는 정책 결정자들에게 도전과제를 제시하며, 적절한 대책을 마련하는 데 어려움을 겪게 할 수 있습니다.'

In [7]:
# <배치(Batch) 출력>

# 주어진 주제 리스트를 배치로 출력
chain.batch([{"topic": "더블딥"}, {"topic": "인플레이션"}])

['더블딥(double dip)은 경제학에서 사용되는 용어로, 경기침체가 두 번 발생하는 현상을 의미합니다. 일반적으로 경제가 한 번 침체된 후에 잠시 회복의 기미를 보이다가 다시 침체에 들어가는 상황을 가리킵니다. 이 경우, 일시적인 회복이 지속되지 않고 다시 불황으로 돌아가는 것이 특징입니다. 더블딥은 경제정책의 효과와 소비자 신뢰, 글로벌 경제 상황 등 다양한 요소에 영향을 받습니다.',
 '인플레이션은 일반적인 가격 수준이 지속적으로 상승하는 현상으로, 화폐의 구매력이 감소하는 것을 의미합니다. 이는 다양한 요인에 의해 발생할 수 있으며, 수요 증가, 생산 비용 상승, 통화 공급 확대 등이 그 원인이 될 수 있습니다. 인플레이션이 높으면 생활비가 증가하고 경제 전반에 부정적인 영향을 미칠 수 있지만, 적정 수준의 인플레이션은 경제 성장에 긍정적인 역할을 할 수도 있습니다. 중앙은행은 금리 조정 등을 통해 인플레이션을 관리하려고 노력합니다.']

In [8]:
# <스트림(Stream) 출력>

# 응답을 토큰 단위로 스트리밍하여 출력
for token in chain.stream({"topic":"더블딥"}):
    # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며, 버퍼를 즉시 플러시하여 실시간으로 보여줌
    print(token, end="", flush=True)


더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 두 번 연속 발생하는 상황을 의미합니다. 일반적으로 경제가 한 번 침체에 들어갔다가 일정 기간 회복의 조짐을 보이더니 다시 침체에 빠지는 경우를 가리킵니다. 이 현상은 경제 회복이 불안정하거나, 외부 요인으로 인해 지속적인 성장에 실패하는 경우에 나타날 수 있습니다. 더블딥은 종종 실업률 증가, 소비 감소, 기업 투자 축소 등과 관련되어 나타납니다.

In [9]:
# <구성된 체인을 다른 러너블과 결합하기>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# "이 대답을 영어로 번역해 주세요"라는 질문을 생성하는 프롬프트 템플릿을 정의
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 이전에 정의된 체인(chain)을 사용하여 대답을 생성하고, 그 대답을 영어로 번역하도록 프롬프트에 전달한 후, 모델을 통해 결과를 생성하여 최종적으로 문자열로 파싱하는 체인을 구성
composed_chain = {"answer": chain} | analysis_prompt | model | StrOutputParser()
# "더블딥"이라는 주제로 대답을 생성하고, 체인 실행
composed_chain.invoke({"topic": "더블딥"})

'The term "Double Dip" is used in economics to describe the phenomenon of experiencing two recessions. Typically, it refers to a situation where, after a recession, the economy appears to have recovered, but this recovery is not sustained, leading to a further decline in the economy. This phenomenon usually occurs when economic indicators temporarily improve before worsening again. It is more likely to happen when business and consumer confidence weaken and the labor market becomes unstable. Double dips present a significant challenge for economic policymakers.'

In [10]:
# <람다 함수를 사용한 체인을 통해 구성하기>

# 이전에 정의된 값들
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
chain = prompt | model | StrOutputParser()
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 람다 함수를 사용한 체인 구성
composed_chain_with_lambda = (
    # 이전에 정의된 체인(chain)을 사용하여 입력된 데이터를 받아옵니다.
    chain
    # 입력된 데이터를 "answer" 키로 변환하는 람다 함수를 적용합니다.
    | (lambda input: {"answer": input})
    # "answer " 키를 가진 데이터를 영어로 번역하도록 프롬프트에 전달합니다.
    | analysis_prompt
    # 프롬프트에서 생성된 요청을 모델에 전달하여 결과를 생성합니다.
    | model
    # 모델에서 반환된 결과를 문자열로 파싱합니다.
    | StrOutputParser()
)
# "더블딥"라는 주제로 대답을 생성하고, 그 대답을 영어로 번역합니다.
composed_chain_with_lambda.invoke({"topic": "더블딥"})


'The term "Double Dip" in economics describes a situation where an economy experiences two recessions. It refers to a scenario where, after the first recession ends, the economy appears to be recovering, but then another recession occurs. This can happen due to various factors such as financial instability or a decline in consumer confidence. The double dip phenomenon primarily occurs in economies with weak resilience and poses a significant challenge for policymakers.'

In [11]:
# <`.pipe()`를 통해 체인 구성하기>

# (방법1) 여러 작업을 순차적으로 .pipe를 통해 연결하여 체인 구성하기
composed_chain_with_pipe = (
  # 이전에 정의된 체인(chain)으로 입력된 데이터를 받아옴
  chain
  # 입력된 데이터를 "answer" 키로 변환하는 람다 함수 적용
  .pipe(lambda input: {"answer": input})
  # analysis_prompt를 체인에 연결하여 설명을 영어로 번역하는 작업 추가
  .pipe(analysis_prompt)
  # 모델을 사용해 응답 생성
  .pipe(model)
  # 생성된 응답을 문자열로 파싱
  .pipe(StrOutputParser())
)
# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})


'The term "Double Dip" is used in economics to refer to a situation where a recession occurs twice. It describes a scenario in which the economy does not fully recover after the first recession and instead falls back into another downturn. This phenomenon indicates instability in the economy and is often accompanied by negative indicators such as repeated increases in unemployment and decreases in consumer spending. A double dip presents a significant challenge for policymakers.'

In [12]:
# (방법2) 좀 더 간단하게 연결하기
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())

# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})

'The term "Double Dip" refers to a phenomenon in economics or finance where a recession occurs twice. It describes a situation in which there is not a complete recovery from the first economic downturn before a second downturn occurs. This typically applies when economic indicators show signs of recovery only to decline again, negatively impacting business and consumer confidence. Such a situation can also create anxiety among investors.'

In [13]:
# <`RunnableParallel`을 이용한 체인 구성>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧은 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥은 두 개의 deep learning 모델을 통합하여 하나의 더 우수한 모델을 만드는 기술이다. 주로 앙상블 학습 기법 중 하나로 사용되며, 각 모델이 다른 측면에서 좋은 성능을 보이는 경우에 특히 효과적이다. 더블딥은 예측력을 향상시키고 overfitting을 줄이는 데 도움을 줄 수 있다.
영어 설명: Double deep learning (DoubleDIP) is a novel deep learning method that simultaneously optimizes a target neural network and a regularizer network to improve generalization performance. It aims to find more robust and universal representations by leveraging the complementary strengths of both networks.


In [14]:
# <`RunnableParallel`를 이용한 체인 구성하기>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥이란 인터넷 슬랭으로, 두 번 깊게 파고들어서 자세히 조사하거나 탐구하는 것을 의미합니다. 주로 어려운 문제나 복잡한 주제를 깊이 연구하거나 이해하기 위해 사용되는 용어로 쓰입니다.
영어 설명: Double-dipping is the act of putting a chip or other food item back into a communal dip after taking a bite. It is considered unsanitary and impolite in many cultures.


In [15]:
# 참고
class CustomLCEL:
    def __init__(self, value):
        self.value = value  # 객체 생성 시 값을 초기화합니다.
    def __or__(self, other):
        if callable(other):
            return CustomLCEL(other(self.value))  # other가 함수일 경우, 함수를 호출하고 그 결과를 새로운 객체로 반환합니다.
        else:
            raise ValueError("Right operand must be callable")  # other가 함수가 아니면 에러를 발생시킵니다.
    def result(self):
        return self.value  # 현재 값을 반환합니다.
# 문자열 끝에 느낌표를 추가하는 함수
def add_exclamation(s):
    return s + "!"
# 문자열을 뒤집는 함수
def reverse_string(s):
    return s[::-1]
# 파이프라인을 생성하여 순차적으로 문자열 변환 작업을 수행합니다.
custom_chain = (
    CustomLCEL("랭체인 공부하기")  # "랭체인 공부하기"로 초기화된 객체 생성
    | add_exclamation  # 느낌표 추가
    | reverse_string  # 문자열 뒤집기
)
# 최종 결과를 출력합니다.
result = custom_chain.result()
print(result)
# 출력: !기하부공 인체랭


!기하부공 인체랭
